# 04 Artifact Annotation

This notebook marks bad time segments on filtered raw data.

Current goal:

```text
desc-filtered_meg.fif
  → inspect filtered data interactively
  → mark bad time segments as BAD_* annotations
  → save annotation derivative
```

Output example:

```text
derivatives/meeg-pipeline/sub-1409/meg/qc/
  sub-1409_task-chords_desc-badsegments_annotations.fif
```

This notebook should not contain large processing logic itself. It should call reusable functions from `meeg_pipeline.annotations`.


## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import mne
import pandas as pd

from meeg_pipeline.annotations import (
    apply_bad_annotations,
    load_bad_annotations,
    make_bad_annotations_path,
    prepare_raw_for_bad_segment_annotation,
    plot_for_bad_segment_annotation,
    save_or_load_bad_annotations,
)
from meeg_pipeline.config import load_config
from meeg_pipeline.preprocessing import load_filtered_raw, make_filtered_raw_path
from meeg_pipeline.qc import save_or_load_bad_channels
from meeg_pipeline.workflow import (
    annotation_policy_for_step,
    bad_channels_policy_for_step,
    applied_bad_annotations_status_to_dataframe,
    bad_annotations_status_to_dataframe,
    existing_output_policy_for_step,
    find_recording,
    iter_recordings,
    recording_label,
    recording_path_status_to_dataframe,
    selected_recordings_to_dataframe,
    should_overwrite,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## Interactive plotting backend

Run this if you want MNE to open a separate interactive window.

If your backend already works, this cell can stay as-is.


In [ ]:
%matplotlib qt

mne.viz.set_browser_backend("qt")

print("MNE browser backend:", mne.viz.get_browser_backend())


## Selection

Use single values, lists, `None`, or `"all"`.

Examples:

```python
SUBJECTS = "1409"
SUBJECTS = ["1409", "2827"]
SUBJECTS = "all"

TASKS = "chords"
TASKS = ["chords", "nochords"]
TASKS = "all"
TASKS = ["all"]
```

If the project has no sessions or runs, keep `SESSIONS = None` and `RUNS = None`.


In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

# Single-file/manual inspection cells at the end of notebooks are disabled by default
# so batch runs over many participants do not stop for plots or ad-hoc file views.
RUN_SINGLE_FILE_INSPECTIONS = False
selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)


## Overwrite policy

This notebook writes bad-segment annotation derivatives and can also persist bad-channel changes made in the artifact-annotation browser.

By default, existing annotation files are **not** overwritten. Existing annotation decisions are loaded and the GUI is not opened.

Bad-channel changes made while the annotation GUI is open are saved automatically to the same bad-channel decision file used by notebook 02 and are propagated to `channels.tsv`. If an existing bad-channel decision changes, the bad-channel file is overwritten for that recording.

Allowed values:

```python
OVERWRITE_STEPS = []                 # load existing annotations, inspect only missing ones
OVERWRITE_STEPS = ["annotations"]    # overwrite annotation derivatives
OVERWRITE_STEPS = ["bad_channels"]   # overwrite bad-channel decisions when explicitly saved
OVERWRITE_STEPS = "all"              # overwrite all writing steps
```


In [ ]:
OVERWRITE_STEPS = []

overwrite_settings = pd.DataFrame(
    [
        {
            "step": "annotations",
            "overwrite": should_overwrite("annotations", OVERWRITE_STEPS),
            "policy": annotation_policy_for_step(
                "annotations",
                OVERWRITE_STEPS,
            ),
        },
        {
            "step": "bad_channels",
            "overwrite": should_overwrite("bad_channels", OVERWRITE_STEPS),
            "policy": bad_channels_policy_for_step(
                "bad_channels",
                OVERWRITE_STEPS,
            ),
        },
    ]
)

overwrite_settings


## Check filtered inputs

Artifact annotation is performed on filtered raw derivatives.

This table checks whether the required `desc-filtered_meg.fif` files exist.


In [ ]:
recording_path_status_to_dataframe(
    config,
    selected_recordings,
    make_filtered_raw_path,
    exists_column="filtered_exists",
    path_column="filtered_path",
)


## Bad-segment annotation progress before inspection

This table shows which selected recordings already have saved bad-segment annotations.


In [ ]:
bad_annotations_status_to_dataframe(config, selected_recordings)


## Run interactive artifact annotation

This cell iterates through all `selected_recordings`.

For each recording:

1. The filtered raw derivative is loaded.
2. Existing bad-channel decisions are loaded for comparison.
3. The MNE browser opens.
4. Mark bad time segments as annotations, preferably with descriptions starting with `BAD`.
5. If additional bad channels become visible, mark them in the same browser.
6. Close the browser window.
7. The notebook saves or loads the annotation derivative.
8. The notebook also saves changed bad-channel decisions to the same files used by notebook 02 and updates `channels.tsv`.
9. The next recording starts.

If an annotation file already exists and `OVERWRITE_STEPS = []`, the GUI is not opened and the existing annotations are loaded.


In [ ]:
annotation_policy = annotation_policy_for_step(
    "annotations",
    OVERWRITE_STEPS,
)

bad_channels_policy = bad_channels_policy_for_step(
    "bad_channels",
    OVERWRITE_STEPS,
)

annotation_results = []

for index, recording in enumerate(selected_recordings):
    label = recording_label(recording)

    annotations_path = make_bad_annotations_path(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
    )

    print("=" * 80)
    print(f"Artifact annotation {index + 1}/{len(selected_recordings)}: {label}")
    print("=" * 80)

    existing_annotations = load_bad_annotations(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
    )

    if existing_annotations.annotations is not None and annotation_policy == "load":
        annotation_results.append(
            {
                "index": index,
                "recording": label,
                "status": "loaded_existing",
                "message": "Annotation file already exists; GUI not opened.",
                "n_annotations": existing_annotations.n_annotations,
                "n_bad_annotations": existing_annotations.n_bad_annotations,
                "descriptions": ", ".join(existing_annotations.descriptions or []),
                "annotation_path": existing_annotations.path,
                "bad_channels_status": "not_inspected",
                "bad_channels": "",
                "bad_channels_path": "",
            }
        )

        print("Existing annotation decision found; GUI not opened.")
        print(f"Annotations: {existing_annotations.n_annotations}")
        print(f"BAD annotations: {existing_annotations.n_bad_annotations}")
        print()
        continue

    filtered_result = load_filtered_raw(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        preload=True,
    )

    if filtered_result.raw is None:
        annotation_results.append(
            {
                "index": index,
                "recording": label,
                "status": filtered_result.status,
                "message": filtered_result.message,
                "n_annotations": None,
                "n_bad_annotations": None,
                "descriptions": "",
                "annotation_path": str(annotations_path),
                "bad_channels_status": "not_inspected",
                "bad_channels": "",
                "bad_channels_path": "",
            }
        )

        print(f"Missing filtered input: {filtered_result.path}")
        print()
        continue

    raw_annot = filtered_result.raw

    bads_before = sorted(str(channel) for channel in raw_annot.info["bads"])

    prepare_raw_for_bad_segment_annotation(
        raw_annot,
        keep_existing_bad_annotations=True,
    )

    print("Annotation descriptions before opening GUI:")
    print(sorted(set(str(desc) for desc in raw_annot.annotations.description)))
    print("Bad channels before opening GUI:")
    print(bads_before)

    plot_for_bad_segment_annotation(
        raw_annot,
        picks="meg",
        block=True,
    )

    bads_after = sorted(str(channel) for channel in raw_annot.info["bads"])
    bad_channels_changed = bads_after != bads_before

    bad_channels_on_existing = (
        "overwrite" if bad_channels_changed else bad_channels_policy
    )

    bad_channels_result = save_or_load_bad_channels(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        bads=bads_after,
        method="manual_mne_gui_during_artifact_annotation",
        notes=(
            "Bad channels were reviewed and optionally updated during "
            "interactive artifact annotation with raw.plot(block=True)."
        ),
        on_existing=bad_channels_on_existing,
        update_channels_tsv=True,
    )

    if bad_channels_changed:
        print("Bad channels changed during artifact annotation:")
        print(f"Before: {bads_before}")
        print(f"After:  {bads_after}")
        print(f"Saved to: {bad_channels_result.path}")
    else:
        print("Bad channels unchanged during artifact annotation.")

    result = save_or_load_bad_annotations(
        config,
        subject=recording["subject"],
        session=recording["session"],
        task=recording["task"],
        run=recording["run"],
        annotations=raw_annot.annotations,
        on_existing=annotation_policy,
    )

    annotation_results.append(
        {
            "index": index,
            "recording": label,
            "status": result.status,
            "message": result.message,
            "n_annotations": result.n_annotations,
            "n_bad_annotations": result.n_bad_annotations,
            "descriptions": ", ".join(result.descriptions or []),
            "annotation_path": result.path,
            "bad_channels_status": bad_channels_result.status,
            "bad_channels": ", ".join(bad_channels_result.bads),
            "bad_channels_path": bad_channels_result.path,
        }
    )

annotation_results_table = pd.DataFrame(annotation_results)
annotation_results_table


## Optional inspection

Use this optional section to inspect one filtered recording with saved annotations applied.

Select a recording explicitly by subject/task/session/run. This keeps batch processing above flexible with `SUBJECTS = "all"` and `TASKS = "all"`, while making manual inspection easy.

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    INSPECT_SUBJECT = "1409"
    INSPECT_SESSION = None
    INSPECT_TASK = "chords"
    INSPECT_RUN = None

    INSPECT = find_recording(
        selected_recordings,
        subject=INSPECT_SUBJECT,
        session=INSPECT_SESSION,
        task=INSPECT_TASK,
        run=INSPECT_RUN,
    )

    if INSPECT is None:
        raw_inspect = None

        inspect_status = pd.DataFrame(
            [
                {
                    "recording": "",
                    "filtered_status": "not_selected",
                    "annotation_status": "",
                    "message": (
                        "No matching recording found in selected_recordings. "
                        "Check INSPECT_SUBJECT/SESSION/TASK/RUN or the selection above."
                    ),
                    "filtered_path": "",
                    "annotations_path": "",
                }
            ]
        )

    else:
        filtered_result = load_filtered_raw(
            config,
            subject=INSPECT["subject"],
            session=INSPECT["session"],
            task=INSPECT["task"],
            run=INSPECT["run"],
            preload=False,
        )

        if filtered_result.raw is None:
            raw_inspect = None

            inspect_status = pd.DataFrame(
                [
                    {
                        "recording": recording_label(INSPECT),
                        "filtered_status": filtered_result.status,
                        "annotation_status": "",
                        "message": filtered_result.message,
                        "filtered_path": filtered_result.path,
                        "annotations_path": "",
                    }
                ]
            )

        else:
            raw_inspect = filtered_result.raw

            apply_result = apply_bad_annotations(
                raw_inspect,
                config,
                subject=INSPECT["subject"],
                session=INSPECT["session"],
                task=INSPECT["task"],
                run=INSPECT["run"],
            )

            inspect_status = pd.DataFrame(
                [
                    {
                        "recording": recording_label(INSPECT),
                        "filtered_status": filtered_result.status,
                        "annotation_status": apply_result.status,
                        "message": apply_result.message,
                        "n_annotations": apply_result.n_annotations,
                        "n_bad_annotations": apply_result.n_bad_annotations,
                        "filtered_path": filtered_result.path,
                        "annotations_path": apply_result.path,
                    }
                ]
            )

    inspect_status
else:
    print('Skipped single-file inspection cell 16 in 1B_meg_preprocessing/04_artifact_annotation.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if raw_inspect is None:
        print("No recording loaded. Nothing to plot.")
    else:
        raw_inspect.plot(
            picks="meg",
            block=True,
        )
else:
    print('Skipped single-file inspection cell 17 in 1B_meg_preprocessing/04_artifact_annotation.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    applied_bad_annotations_status_to_dataframe(config, selected_recordings)
else:
    print('Skipped single-file inspection cell 18 in 1B_meg_preprocessing/04_artifact_annotation.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


## Inspect one annotated recording

Use this optional section to inspect one filtered recording with saved annotations applied.

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    INSPECT_INDEX = 0

    INSPECT = selected_recordings[INSPECT_INDEX]

    filtered_result = load_filtered_raw(
        config,
        subject=INSPECT["subject"],
        session=INSPECT["session"],
        task=INSPECT["task"],
        run=INSPECT["run"],
        preload=False,
    )

    raw_inspect = filtered_result.raw

    if raw_inspect is None:
        pd.DataFrame(
            [
                {
                    "recording": recording_label(INSPECT),
                    "filtered_status": filtered_result.status,
                    "annotation_status": "skipped",
                    "message": filtered_result.message,
                    "filtered_path": filtered_result.path,
                }
            ]
        )
    else:
        annotation_result = apply_bad_annotations(
            raw_inspect,
            config,
            subject=INSPECT["subject"],
            session=INSPECT["session"],
            task=INSPECT["task"],
            run=INSPECT["run"],
        )

        pd.DataFrame(
            [
                {
                    "recording": recording_label(INSPECT),
                    "filtered_status": filtered_result.status,
                    "annotation_status": annotation_result.status,
                    "message": annotation_result.message,
                    "n_annotations": annotation_result.n_annotations,
                    "n_bad_annotations": annotation_result.n_bad_annotations,
                    "filtered_path": filtered_result.path,
                    "annotations_path": annotation_result.path,
                }
            ]
        )
else:
    print('Skipped single-file inspection cell 20 in 1B_meg_preprocessing/04_artifact_annotation.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


### Optional interactive inspection

Run this only when using a local interactive backend.

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if raw_inspect is None:
        print("No recording loaded. Nothing to plot.")
    else:
        raw_inspect.plot(block=True)
else:
    print('Skipped single-file inspection cell 22 in 1B_meg_preprocessing/04_artifact_annotation.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')
